In [1]:
# Cập nhật danh sách gói (bắt buộc)
!sudo apt-get update

# Cài đặt công cụ zstd để giải nén file .tar.zst của Ollama phiên bản mới
!sudo apt-get install -y zstd

# Cài đặt các thư viện Python
%pip install fastapi uvicorn ollama

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]      
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,990 kB]


In [2]:
# Chạy script cài đặt Ollama (lúc này đã có zstd để giải nén)
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [71]:
import subprocess
import time

# (Tùy chọn) Kill tiến trình ollama cũ nếu có, để tránh xung đột cổng
subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

# Khởi động ollama serve trong background
# stdout và stderr được chuyển hướng để tránh làm lộn xộn output của notebook
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi vài giây cho server khởi động hoàn toàn
time.sleep(4)
print("✅ Ollama server đã được khởi động thành công!")


✅ Ollama server đã được khởi động thành công!


In [4]:
import ollama
ollama.pull('hf.co/bartowski/Llama-3.1_OpenScholar-8B-GGUF:Q6_K')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [ ]:
# Kiểm tra xem server đã sẵn sàng chưa




# Tải model Qwen3:8b về (chỉ cần làm một lần)
!ollama pull qwen3:0.6b
!ollama pull qwen3:1.7b
!ollama pull qwen3:4b

]11;?\NAME                                                  ID              SIZE      MODIFIED      
hf.co/bartowski/Llama-3.1_OpenScholar-8B-GGUF:Q6_K    61af2317513a    6.6 GB    7 seconds ago    
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 7f4030143c1c:   0% ▕                  ▏ 220 KB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  10% ▕█                 ▏  50 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  17% ▕███               ▏  90 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  31% ▕█████             ▏ 164 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  47% ▕████████          ▏ 246 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  54% ▕█████████         ▏ 284 MB/522 MB                  pulling manifest 
pulling 7f4030143c1c:  69% ▕████████████      ▏ 358 MB/522

In [3]:
!ollama list

]11;?\NAME                                                  ID              SIZE      MODIFIED      
qwen3:4b                                              359d7dd4bcda    2.5 GB    4 minutes ago    
qwen3:1.7b                                            8f68893c685c    1.4 GB    5 minutes ago    
qwen3:0.6b                                            7df6b6e09427    522 MB    5 minutes ago    
hf.co/bartowski/Llama-3.1_OpenScholar-8B-GGUF:Q6_K    61af2317513a    6.6 GB    5 minutes ago    


In [43]:
# Cell 5: Hàm gọi model (thay thế cho API)
from ollama import generate
from pydantic import BaseModel
from pathlib import Path

class Response(BaseModel):
    qtype: str


def load_prompt(filename: str,path: str, **kwargs) -> str:
    prompt_path = Path(path+ filename)
    template = prompt_path.read_text(encoding="utf-8")
    # Nếu có placeholder, thay thế bằng kwargs
    for key, value in kwargs.items():
        placeholder = "{{" + key + "}}"
        template = template.replace(placeholder, value)
    return template


def classify_with_llm(question: str) :
    prompt_text = load_prompt(filename="prompt.txt",path="/kaggle/input/datasets/nam1706/prompt-v1-2/", QUESTION="QUESTION: " + question)
    response = generate(
        model='qwen3:0.6b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'think':False,
            'stream':False,
            'format': Response.model_json_schema(),
            'keep_alive':'5m',
            'num_predict': 1000  # CHẶN: Không cho sinh quá 100 token
        }
    )
    # ... gọi ollama với prompt trên, temperature=0
    raw = response # giả sử đây là string trả về
    # LLM thường trả về markdown json```json\n{...}\n```, cần làm sạch
    # Dùng regex tìm khối JSON đầu tiên
    return raw['response']

In [25]:

q0= "Based on the premises, what can we conclude about the curriculum?\nA. It enhances student engagement but not critical thinking\nB. It enhances critical thinking\nC. It needs more resources to enhance critical thinking\nD. It is well-structured but lacks exercises"+"\nDoes the combination of faculty priorities and curriculum features lead to enhanced critical thinking?"

q1 = "Two electric charges q1 = +9×10⁻⁶ C and q2 = -9×10⁻⁶ C are placed 10 cm apart. A charge q3 = +9×10⁻⁶ C is at the midpoint. Bring the problem to SMT formula to find forces between q1 and q3"

q2 = "Calculate the energy stored in capacitor C when C = 100 μF and U = 30 V."

q3 = "calculate "

q4 = "Calculate the current through a 10 Ohm resistor connected in series with a 20 Ohm resistor to a 12V battery."
print(classify_with_llm(q1))


{
  "qtype": "physics"
}


In [79]:
from ollama import generate
from pydantic import BaseModel
from pathlib import Path

class Response(BaseModel):
    qtype: str


def problem_extracter(question: str):
    prompt_text = load_prompt(filename="physicPrompt.txt",path="/kaggle/input/datasets/nam1706/phyprompt-vfinal/", QUESTION="QUESTION: " + question)
    response = generate(
        model='qwen3:1.7b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'think':False,
            'stream':False,
            'format': Response.model_json_schema(),
            'keep_alive':'5m',
            'num_predict': 6000  # CHẶN: Không cho sinh quá 100 token
        }
    )
    raw = response
    return raw['response']



In [83]:
q2 = "Calculate the energy stored in capacitor C when C = 100 μF and U = 30 V."
q0 = "An object undergoes simple harmonic motion with the equation x = 4cos(2πt + π/2) cm. Determine the amplitude, period, and initial position of the object."
q3_diff2 = "In a circuit, a 10 Ohm resistor (R1) and a 15 Ohm resistor (R2) are connected in series. This combination is then connected in series with a 4 Ohm resistor (R3) and a 24 V battery. Calculate the total current flowing from the battery and the voltage drop across R2."
q6 = "A 5 Ohm and a 10 Ohm resistor are connected in series to a 15 V battery. Find the current."
q4 = "In a circuit, a 6 Ohm resistor (R1) and a 12 Ohm resistor (R2) are connected in parallel. This combination is then connected in series with a 10 Ohm resistor (R3) and a 36 V battery. Calculate the total current and the power dissipated by R3."
q5 = "A car accelerates uniformly from rest to 20 m/s in 8 seconds. It then accelerates again at 1.5 m/s² for 6 seconds. Calculate the final velocity of the car and the total distance traveled."
q7 = "In a circuit, a 6 Ohm (R1) and a 12 Ohm (R2) are in parallel. This block is in series with a 10 Ohm (R3) and a 36 V source. Find the current through the source and the voltage across R2."
q8 = "How much heat is needed to heat 300 g of aluminum from 25°C to 75°C? Specific heat of aluminum is 900 J/(kg·K)."
q9 = "A stone is dropped from rest. How far does it fall in 4 seconds? (Take g = 10 m/s)"
q10 = "A 5 kg block on a frictionless 30° incline is connected to a 3 kg hanging mass by a string over a pulley. Find the acceleration and the tension in the string."

q11 = "In a circuit, a 10 V battery (E1) with internal resistance 1 Ohm, and a 6 V battery (E2) with internal resistance 0.5 Ohm are connected in parallel. The combination powers a load resistor R = 10 Ohm. Find the current through the load resistor."
q12 = "In a circuit, a 12 V battery (E1) with an internal resistance of 2 Ω, and a 6 V battery (E2) with an internal resistance of 1 Ω are connected in parallel. This parallel combination of batteries is then used to power a load resistor R = 10 Ω connected across the combination. Determine the current flowing through the load resistor R, and the current supplied by each battery."

q13 ="A 5 kg block lies on a rough 37° incline (coefficient of kinetic friction µ_k = 0.2). This block is connected by a light string passing over a frictionless pulley to a hanging 8 kg mass. The system is released from rest. Determine the acceleration of the system and the tension in the string. (Take g = 10 m/s², sin37° = 0.6, cos37° = 0.8)."
q14 = "A 0.5 kg iron block at 300°C is dropped into 2 kg of water at 25°C. Assuming no heat loss, find the final equilibrium temperature. (c_iron = 450 J/kg·K, c_water = 4186 J/kg·K)"
q15 = "A car accelerates uniformly from rest to 20 m/s in 8 seconds. It then accelerates again at 1.5 m/s² for 6 seconds. Calculate the final velocity of the car and the total distance traveled."
q16 = "An object undergoes simple harmonic motion with maximum speed vmax = 16π (cm/s) and maximum acceleration amax = 6.4 (m/s²). Take π² = 10. Calculate the period and frequency of oscillation."
print(problem_extracter(q16))

{
  "given": [
    {"name": "maximum speed", "symbol": "vmax", "value": 16, "unit": "cm/s"},
    {"name": "maximum acceleration", "symbol": "amax", "value": 6.4, "unit": "m/s²"},
    {"name": "π²", "symbol": "π²", "value": 10, "unit": null}
  ],
  "unknown": [
    {"name": "period", "symbol": "T", "value": null, "unit": "s"},
    {"name": "frequency", "symbol": "f", "value": null, "unit": "1/s"}
  ],
  "topology": "simple_harmonic_motion",
  "problem_type": "dynamics",
  "difficulty": 2
}


In [78]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:5])

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/nam1706 []
/kaggle/input/datasets/nam1706/phyprompt-vfinal ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-1-2-backup ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-2 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/prompt-v1-2 ['prompt.txt']
/kaggle/input/datasets/nam1706/physicprompt-v1-0 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-4 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-6 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-5 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-3 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-7 ['physicPrompt.txt']
/kaggle/input/datasets/nam1706/phyprompt-v1-1-back-up ['physicPrompt.txt']
